# Zurich landing trajectories — data & fPCA exploration

Quick look at the processed dataset and the fPCA latent space that the DDPM is trained on.
Run with the traffic-capable venv, from the repo root or from `notebooks/`.

```bash
/Users/meldor/Desktop/git/deep-traffic-generation-paper/.venv/bin/python -m jupyter lab
```

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))  # repo root when running from notebooks/
sys.path.insert(0, os.path.abspath('.'))
import numpy as np
import matplotlib.pyplot as plt
from src.pipeline.utils import load_config
from src.data.dataset import load_dataset
from src.fpca import FPCA

cfg = load_config()  # resolves to <repo>/configs/config.yaml
data = load_dataset(cfg)  # runs Stage 1 cache; needs data/processed.npz
X, names = data['X'], data['feature_names']
print('X', X.shape, '| features', names)
print('flows:', dict(zip(*np.unique(data['flow'].astype(str), return_counts=True))))

## Feature profiles
Mean +/- std of each channel across the 200-step approach (raw units).

In [ ]:
t = np.arange(X.shape[1])
fig, axes = plt.subplots(1, len(names), figsize=(3.4*len(names), 3.0))
for ax, j, name in zip(axes, range(len(names)), names):
    mu, sd = X[:, :, j].mean(0), X[:, :, j].std(0)
    ax.plot(t, mu, color='#1f77b4'); ax.fill_between(t, mu-sd, mu+sd, alpha=0.2, color='#1f77b4')
    ax.set_title(name); ax.set_xlabel('timestep'); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## Spatial view
Real lat/lon tracks, coloured by approach flow — note convergence to the Zurich FAF.

In [ ]:
ll = data['latlon']; flow = data['flow'].astype(str)
rng = np.random.default_rng(0); idx = rng.choice(len(ll), 600, replace=False)
flows = sorted(set(flow[idx]))
cmap = plt.get_cmap('tab10')
plt.figure(figsize=(7, 6))
for k, fl in enumerate(flows):
    sel = idx[flow[idx] == fl]
    for s in sel[:120]:
        plt.plot(ll[s, :, 1], ll[s, :, 0], color=cmap(k % 10), lw=0.4, alpha=0.4)
    plt.plot([], [], color=cmap(k % 10), label=fl)
plt.legend(title='initial_flow', fontsize=8); plt.xlabel('longitude'); plt.ylabel('latitude')
plt.gca().set_aspect('equal', 'box'); plt.title('Real approaches into LSZH'); plt.show()

## fPCA latent space
Fit the per-feature bases and inspect the scree curves and the 2D score cloud.

In [ ]:
fpca = FPCA.fit(data['X_std'][data['train_idx']], names,
                explained_variance=cfg['fpca']['explained_variance'],
                max_components=cfg['fpca']['max_components'])
print('components/feature:', dict(zip(names, fpca.ks)), '-> m =', fpca.m)
print('explained variance:', {k: round(v, 4) for k, v in fpca.total_explained_variance().items()})

fig, axes = plt.subplots(1, len(names), figsize=(3.2*len(names), 2.8))
for ax, name, b in zip(axes, names, fpca.bases):
    cum = np.cumsum(b['evr_full'])
    ax.plot(np.arange(1, len(cum)+1), cum, marker='o', ms=3)
    ax.axvline(b['k'], color='r', ls='--', lw=1); ax.set_ylim(0, 1.02)
    ax.set_xlim(0.5, b['k']+6); ax.set_title(name); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# 2D score cloud (first two groundspeed components), coloured by flow
W = fpca.encode(data['X_std'])
gs_slice = fpca.slices[names.index('groundspeed')]
s = W[:, gs_slice]
flow_all = data['flow'].astype(str)
plt.figure(figsize=(6, 5))
for k, fl in enumerate(sorted(set(flow_all))):
    m = flow_all == fl
    plt.scatter(s[m, 0], s[m, 1], s=4, alpha=0.3, color=plt.get_cmap('tab10')(k % 10), label=fl)
plt.legend(title='flow', fontsize=8); plt.xlabel('gs PC1'); plt.ylabel('gs PC2')
plt.title('fPCA scores are multimodal (5 approach flows)'); plt.show()